In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
cifar_transform = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                      transforms.Resize((28, 28)),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.5,), (0.5,))])

mnist_transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

In [3]:
class TaggedDataset(Dataset):
    def __init__(self, dataset, tag):
        self.dataset = dataset
        self.tag = tag

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img, label, self.tag  # tag: 0 for MNIST, 1 for CIFAR

MNIST Tag is 0
CIFAR10 Tag is 1

In [4]:
mnist = TaggedDataset(datasets.MNIST(root='./data', train=True, download=True,
                            transform=mnist_transform), tag=0)

cifar = TaggedDataset(datasets.CIFAR10(root='./data', train=True, download=True,
                              transform=cifar_transform), tag=1)

Files already downloaded and verified


In [5]:
combined_dataset = ConcatDataset([mnist, cifar])

aşağıda random split yapmak mı daha iyi olur? yoksa acaba hepsi için ayrı mnist/cifar10 data seti mi oluşturmalıydık?

"mnist_train = torchvision.datasets.MNIST(root='./data', train=True, 
                                           download=True, transform=mnist_transform)
    mnist_test = torchvision.datasets.MNIST(root='./data', train=False, 
                                          download=True, transform=mnist_transform)"

bunun gibi

In [6]:
train_ratio = 0.7
val_raito = 0.15
test_ratio = 0.15

dataset_len = len(combined_dataset)
train_len = int(train_ratio * dataset_len)
val_len = int(val_raito * dataset_len)
test_len = dataset_len - train_len - val_len

dataset_train, dataset_val, dataset_test = random_split(combined_dataset, [train_len, val_len, test_len])

In [7]:
batch_size = 128 #32 de dene!

train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)
#test_loader batch_size=1 ???? ne faydası oluyor sor

In [8]:
class MixedMLP(nn.Module):
    def __init__(self, input_dim=28*28, output_dim=10): #hidden_dim = 256
        super().__init__()
        #self.model = nn.Sequential(
            #nn.Flatten(),
            #nn.Linear(28*28, 256),
            #nn.ReLU(),
            #nn.Linear(256, 128),
            #nn.ReLU(),
            #nn.Linear(128, 10)  # Assuming 10 classes
        #)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        #galiba step eklendiğinde buraya eklenecek.
        self.fc3 = nn.Linear(128, output_dim)
        #self.relu3 = nn.ReLU()
        #batch normalization kullanılabilir!! dene!

    def forward(self, x):
        #return self.model(x)
        x = self.flat(x)
        
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)

        return x

In [9]:
learning_rate = 0.001
patience = 5
epochs = 10

In [10]:
model = MixedMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

running_losslist = []
valid_losslist = []

valid_loss_min = np.inf
patience_counter = 0


for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    valid_loss = 0.0
    train_acc = 0.0
    valid_acc = 0.0
    
    correct = 0
    total = 0

    for inputs, labels, tag in tqdm(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_acc += (predicted == labels).sum().item()

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    model.eval()
    with torch.no_grad():
        for data, target, tag in tqdm(val_loader):
            optimizer.zero_grad()#önceki gradyanları sıfırlar
            data, target = data.to(device), target.to(device)
            output = model(data)# aslında model.forward(data), tahmin üretir

            loss_val = criterion(output, target)#buna neden gerek duyduk ki?
            valid_loss += loss_val.item() * data.size(0)
            _, predicted = torch.max(output.data, 1)
            valid_acc += (predicted == target).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    train_loss = running_loss / len(train_loader.dataset)
    valid_loss = valid_loss / len(val_loader.dataset)

    train_acc = train_acc / len(train_loader.dataset)
    valid_acc = valid_acc / len(val_loader.dataset)


    print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {valid_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {valid_acc:.4f}')
    running_losslist.append(running_loss)
    valid_losslist.append(valid_loss)


    if valid_loss <= valid_loss_min:
        valid_loss_min = valid_loss
        patience_counter = 0
    elif patience_counter < patience:
        patience_counter += 1
    else:
        model.eval()
        print("Early stopping")
        break

100%|██████████| 129/129 [00:12<00:00, 10.32it/s]


Epoch [1/10], Train Loss: 1.1321, Val Loss: 0.9761, Train Acc: 0.6107, Val Acc: 0.6630


100%|██████████| 129/129 [00:11<00:00, 10.89it/s]


Epoch [2/10], Train Loss: 0.9160, Val Loss: 0.8857, Train Acc: 0.6831, Val Acc: 0.6901


100%|██████████| 129/129 [00:03<00:00, 32.51it/s]


...

Epoch [8/10], Train Loss: 0.6564, Val Loss: 0.8179, Train Acc: 0.7711, Val Acc: 0.7208


100%|██████████| 129/129 [00:03<00:00, 32.60it/s]


Epoch [9/10], Train Loss: 0.6339, Val Loss: 0.7985, Train Acc: 0.7795, Val Acc: 0.7266


100%|██████████| 129/129 [00:03<00:00, 32.71it/s]

Epoch [10/10], Train Loss: 0.6101, Val Loss: 0.8085, Train Acc: 0.7865, Val Acc: 0.7295
